# CryoSPARC CTF Plotting

This notebook can be used to visualize micrograph CTFs from CryoSPARC and overlay CTF distributions from multiple data sets.

Minimum inputs:
1. Change CSV_DIR to the directory containing the .csv file(s) that contain CTF information from CryoSPARC. To obtain this file, build a Curate Exposures job, and then click "Download table CSV" (at the top of the micrographs table, next to the Filter dropdown). For ease of labeling graphs, I recommend changing the name of the CSV to something understandable (ie CryoSPARC.csv).
2. Change OUTDIR to the directory where you would like your images to be saved.
3. If you have >4 CTF jobs to compare, you will have to increase the number of colors provided in the set below. Here are all your options: https://stackoverflow.com/questions/22408237/named-colors-in-matplotlib

You can also use this notebook to plot other variables from the Curate Micrographs job. Your other options are:
Avg Inten., DF Avg, Astig, DF Range, Tilt Angle, Rel Ice Thick., Motion dist., Motion curv. If you want to do this, replace all instances of 'CTF Fit' in this notebook with your desired variable.

In [ ]:
import pandas as pd
import glob
import os
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import re
import numpy as np

In [ ]:
## CHANGE ME ##

# Folder containing your CSV files
CSV_DIR = '/path/to/csv/directory'
OUTDIR = '/path/images/save/to'

#Set overall colors -- add more if needed
colors = ('lightsteelblue', 'mediumturquoise', 'wheat', 'lightgray')

In [ ]:
# Find all CSV files in the folder
csv_files = glob.glob(os.path.join(CSV_DIR, "*.csv"))

df_cryosparc = {}  # dictionary to hold dataframes

for fpath in csv_files:
    # Use filename without extension as the dataframe name
    name = os.path.splitext(os.path.basename(fpath))[0]
    
    # Read CSV; adjust parameters if needed
    df = pd.read_csv(fpath)
    
    # Ensure the index column is numeric and usable as x-axis
    # If "Index" is a column, set it as index or keep as column
    if "Index" in df.columns:
        df["Index"] = pd.to_numeric(df["Index"], errors="coerce")
    
    df_cryosparc[name] = df


# Expose them as variables in the notebook's global namespace
globals().update(df_cryosparc)

df_cryosparc.keys()  # show the loaded dataframe names

## Scatterplot / histogram of all .CSVs in the folder

To graph a scatterplot or histogram of all .csvs in your folder, just run the cells. You may have to change the ylim (y-axis min and max) depending on your CTFs.

In [ ]:
# Scatterplot of all .csvs

count = 0
for key in df_cryosparc:
    plt.scatter(df_cryosparc[key].index, df_cryosparc[key]['CTF Fit'], s=1, alpha=0.5, color=colors[count], label=key)
    plt.axhline(df_cryosparc[key]['CTF Fit'].mean(), color=colors[count], linestyle='dashed')
    count = count + 1

# plt.title('Title')
plt.xlabel('Micrograph Index')
plt.ylabel('CTF Fit')
plt.ylim(0,10) # <-- change me if necessary
plt.legend(markerscale=5, loc='upper right')

# png_name = os.path.join(OUTDIR, "CTF_scatter_all.png")
# plt.savefig(png_name)

plt.show()

In [ ]:
# Histogram of all .csvs

bins=np.arange(2, 10, 0.25) # <- change me if you need a different range or bin size

count = 0
for key in df_cryosparc:
    df_cryosparc[key]['CTF Fit'].hist(bins=bins, color=colors[count], alpha=0.5, label=key)
    count = count + 1
    
#plt.title('CTF Histogram')
plt.xlim(2,8) # <-- change me if necessary
plt.xlabel('CTF Fit')
plt.ylabel('Frequency')
plt.grid(False)

# png_name = os.path.join(OUTDIR, "CTF_hist_all.png")
# plt.savefig(png_name)

plt.show()

## Specific .CSVs only

To graph a scatterplot or histogram of only some of the .csvs in a folder, fill in the key_1 and key_2 variables with the file names (everything before the .csv) and then run the cells. If you want to compare >2, just add more keys and follow the pattern to add more series to the scatterplot. Again, you may have to change the ylim (y-axis min and max) depending on your CTFs.

In [ ]:
# Scatterplot of specific .csvs

key_1 = 'fill_me_in'
key_2 = 'fill_me_in'
# etc

plt.scatter(df_cryosparc[key_1].index, df_cryosparc[key_1]['CTF Fit'], s=1, alpha=0.5, color=colors[0], label=key_1)
plt.axhline(df_cryosparc[key_1]['CTF Fit'].mean(), color=colors[0], linestyle='dashed')

plt.scatter(df_cryosparc[key_2].index, df_cryosparc[key_2]['CTF Fit'], s=1, alpha=0.5, color=colors[1], label=key_2)
plt.axhline(df_cryosparc[key_2]['CTF Fit'].mean(), color=colors[1], linestyle='dashed')

#etc

# plt.title('Title')
plt.xlabel('Micrograph Index')
plt.ylabel('CTF Fit')
plt.ylim(2,8) # <-- change me if necessary
plt.legend(markerscale=5, loc='upper right')

# png_name = os.path.join(OUTDIR, "CTF_scatter_X_vs_Y.png")
# plt.savefig(png_name)

plt.show()

In [ ]:
# Histogram of specific .csvs

key_1 = 'fill_me_in'
key_2 = 'fill_me_in'
# etc

bins=np.arange(0, 10, 0.25) # <- change me if you need a different range or bin size

df_cryosparc[key_1]['CTF Fit'].hist(bins=bins, color=colors[1], alpha=0.5, label=key_1)
df_cryosparc[key_2]['CTF Fit'].hist(bins=bins, color=colors[2], alpha=0.5, label=key_2)
#etc.

#plt.title('CTF Histogram')
plt.xlim(0,10) # <-- change me if necessary
plt.xlabel('CTF Fit')
plt.ylabel('Frequency')
plt.grid(False)
plt.legend()

# png_name = os.path.join(OUTDIR, "CTF_hist_X_vs_Y.png")
# plt.savefig(png_name)

plt.show()